# 03 — Complexité cyclomatique de McCabe

**Auteur :** Benoît Moraillon — **Licence :** AGPL-3.0

Proposée par Thomas McCabe en 1976, la **complexité cyclomatique** mesure le **nombre de chemins d'exécution indépendants** dans un fragment de code. Elle est centrale en assurance qualité et en couverture de test.

## Définition

Pour un graphe de contrôle de flux :

$$ CC = E - N + 2P $$

où E = arêtes, N = nœuds, P = composantes connexes (généralement 1 pour une fonction).

**Formule pratique équivalente** utilisée dans KodoNeko :

$$ CC = 1 + \text{nombre de points de décision} $$

Points de décision : `if`, `elif`, `for`, `while`, `case`, `except`, `and` / `or` / `&&` / `||`, ternaires `? :`, compréhensions, etc.

## Seuils standards (NIST, Carnegie Mellon SEI)

| CC | Niveau | Recommandation |
|---|---|---|
| 1-10 | low | Code simple, faible risque |
| 11-20 | moderate | À surveiller |
| 21-50 | high | Risque accru, refactoring conseillé |
| > 50 | very_high | Refactoring fortement conseillé |

Ces seuils sont implémentés dans `mccabe.level`.

In [ ]:
# Bootstrap : permet l'import des modules sans installation pip
import sys
from pathlib import Path

root = Path.cwd().parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import setup_paths
setup_paths.setup()

## Exemple progressif

In [ ]:
from kodoneko_metrics import count_mccabe

source_simple = '''
def identite(x):
    return x
'''

source_avec_branches = '''
def categoriser(age):
    if age < 13:
        return "enfant"
    elif age < 18:
        return "ado"
    elif age < 65:
        return "adulte"
    else:
        return "senior"
'''

for nom, src in [("Identité", source_simple),
                  ("Catégoriser (4 branches)", source_avec_branches)]:
    m = count_mccabe(src, language="python")
    print(f"{nom:30s}  CC = {m.value}  ({m.level})")

**Interprétation** : `identite` ne fait aucune décision (1 chemin) → CC = 1. `categoriser` a 3 `if`/`elif` (3 points de décision) → CC = 4. Chaque branche supplémentaire ajoute +1.

## Les opérateurs booléens comptent

Dans la formule standard McCabe, `and` et `or` ajoutent chacun un point de décision (chacun crée un court-circuit logique = un chemin).

In [ ]:
source_booleens = '''
def autoriser(user):
    if user.is_active and user.email and not user.banned:
        return True
    return False
'''

m = count_mccabe(source_booleens, language="python")
print(f"CC = {m.value}, decisions = {m.decision_count}")

Le `if` apporte 1 + `and` × 2 + `not` (qui ne compte pas) = 3 décisions, CC = 4.

## McCabe vs NCLOC : indépendance

Une fonction longue peut être simple, une fonction courte peut être complexe.

In [ ]:
long_mais_lineaire = '''
def setup():
    db = connect()
    cache = init_cache()
    logger = setup_logger()
    config = load_config()
    server = create_server()
    return server, db, cache, logger, config
'''

court_mais_complexe = '''
def f(a, b, c):
    if (a > 0 and b > 0) or (c < 0 and not a) or (b == c):
        return 1
    return 0
'''

for nom, src in [("Long mais linéaire", long_mais_lineaire),
                  ("Court mais complexe", court_mais_complexe)]:
    from kodoneko_metrics import count_lines
    n = count_lines(src, language="python").ncloc
    m = count_mccabe(src, language="python")
    print(f"{nom:30s}  NCLOC={n:>3}  CC={m.value:>2}")

C'est l'argument central de McCabe : **deux fonctions de taille égale peuvent avoir une testabilité radicalement différente**. CC = nombre minimum de cas de test pour couvrir tous les chemins d'exécution.

## Limite : McCabe ne mesure pas l'imbrication

Un `if` à plat et un `if` au fond de 5 boucles imbriquées coûtent le même +1 en McCabe. Pourtant le second est cognitivement bien plus dur à suivre. C'est la cognitive complexity de Campbell (intégrée à **Understandability**, notebook 04) qui corrige ça.

## API de référence

| Fonction | Retour |
|---|---|
| `count_mccabe(source, language=...)` | `McCabeMetrics(value, decision_count, level)` |

---

# Mesurer sur un dépôt réel, et dans le temps

La complexité cyclomatique compte les chemins d'exécution indépendants — le
nombre de tests qu'il faudrait, au minimum, pour couvrir toutes les branches.
Sur un dépôt entier, elle dessine la carte des zones où le code se ramifie le
plus. Mesurons-la sur un projet réel, puis regardons-la dériver dans le temps.

> ⚠️ Cette section clone un dépôt public (nécessite un accès réseau) et, pour
> les langages autres que Python, requiert `tree-sitter`. Sans ces prérequis,
> les cellules affichent un message explicatif et s'interrompent proprement,
> sans erreur.

## 1. Récupérer un dépôt public

Nous clonons un petit projet open-source réel. Le clone est *idempotent* : si le
dépôt est déjà présent localement, on le réutilise sans le retélécharger.

In [ ]:
import subprocess, tempfile
from pathlib import Path

REPO_URL = "https://github.com/gothinkster/django-realworld-example-app.git"
repo_dir = Path(tempfile.gettempdir()) / "kodoneko_demo_repo"

if repo_dir.exists():
    print("Dépôt déjà présent :", repo_dir)
else:
    print("Clonage en cours...")
    res = subprocess.run(
        ["git", "clone", REPO_URL, str(repo_dir)],
        capture_output=True, text=True,
    )
    if res.returncode == 0:
        print("Cloné :", repo_dir)
    else:
        print("Clone impossible (réseau indisponible ?) :")
        print("  ", res.stderr.strip().splitlines()[-1] if res.stderr.strip() else "erreur inconnue")
        repo_dir = None

## 2. Analyser le dépôt complet (instantané)

Premier réflexe : prendre une photographie de l'état actuel. On mesure la complexité cyclomatique
sur l'ensemble du dépôt, tel qu'il est à l'instant présent (le dernier commit).

In [ ]:
from kodoneko_scanner import scan_repo

if repo_dir:
    report = scan_repo(repo_dir)
    print(f"Fichiers analysés : {report.files_scanned}")
    print(f"  Complexité cyclomatique totale : {report.totals.mccabe_sum}")
    print(f"  Complexité moyenne par fonction : {report.totals.mccabe_avg:.1f}")
    print(f"  Pic de complexité (max)         : {report.totals.mccabe_max}")
else:
    print("(dépôt indisponible — étape ignorée)")

## 3. Analyser un commit précis

Le moteur temporel `kodoneko_temporal` sait reconstituer l'état du dépôt à
**n'importe quelle référence git** (un tag, une branche, un SHA, ou une
expression comme `HEAD~5`). Ici, on mesure complexité cyclomatique à ce point de l'histoire, cinq commits
en arrière — un véritable voyage dans le passé du code.

In [ ]:
from kodoneko_temporal import analyze_at_ref

def mesure(path):
    """Applique notre métrique à un état du dépôt."""
    report = scan_repo(path)
    return report.totals.mccabe_sum

if repo_dir:
    try:
        point = analyze_at_ref(repo_dir, "HEAD~5", analyzer=mesure)
        print(f"À {point.label} (commit {point.sha[:8]}) : complexité cyclomatique = {point.result}")
    except Exception as e:
        print(f"Commit indisponible (historique trop court ?) : {e}")
else:
    print("(dépôt indisponible — étape ignorée)")

## 4. Mesurer un écart entre deux moments (*diff temporel*)

La question la plus parlante n'est pas « combien ? » mais « **combien de plus
qu'avant ?** ». En comparant la mesure à deux références, on quantifie ce qu'un
intervalle de développement a réellement produit comme complexité cyclomatique.

In [ ]:
if repo_dir:
    try:
        avant = analyze_at_ref(repo_dir, "HEAD~10", analyzer=mesure)
        apres = analyze_at_ref(repo_dir, "HEAD", analyzer=mesure)
        delta = apres.result - avant.result
        signe = "+" if delta >= 0 else ""
        # Variation relative en pourcentage (2 décimales), protégée contre la division par zéro
        if avant.result:
            pct = delta / avant.result * 100
            pct_txt = f"{signe}{pct:.2f} %"
        else:
            pct_txt = "n/a (valeur initiale nulle)"
        print(f"complexité cyclomatique il y a 10 commits : {avant.result}")
        print(f"complexité cyclomatique aujourd'hui       : {apres.result}")
        print(f"Variation absolue    : {signe}{delta}")
        print(f"Variation relative   : {pct_txt}")
    except Exception as e:
        print(f"Diff indisponible : {e}")
else:
    print("(dépôt indisponible — étape ignorée)")

### La complexité, cette marée montante

La complexité a tendance à croître silencieusement : chaque `if` ajouté à la
hâte gonfle le total. Suivre la somme cyclomatique mois par mois, c'est garder
l'œil sur cette marée — et repérer le moment où il devient urgent de refactorer.

In [ ]:
from kodoneko_temporal import analyze_over_windows

if repo_dir:
    serie = analyze_over_windows(repo_dir, analyzer=mesure, strategy="monthly")
    print(f"complexité cyclomatique à la fin de chaque mois ({len(serie)} fenêtres) :\n")
    maxv = max((p.result for p in serie), default=1) or 1
    precedent = None
    for point in serie:
        barre = "█" * min(40, int(point.result / maxv * 40))
        # Variation par rapport au mois précédent, en % (2 décimales)
        if precedent is None:
            var_txt = "     —  "
        elif precedent:
            pct = (point.result - precedent) / precedent * 100
            var_txt = f"{pct:+6.2f} %"
        else:
            var_txt = "    n/a "
        print(f"  {point.label}  {point.result:>8}  {var_txt}  {barre}")
        precedent = point.result
else:
    print("(dépôt indisponible — étape ignorée)")

## En résumé

Nous venons de parcourir les quatre façons d'interroger la complexité cyclomatique :

1. **L'instantané** — l'état présent du dépôt entier.
2. **Le commit précis** — la mesure à un point quelconque de l'histoire.
3. **Le diff temporel** — l'écart entre deux moments, qui isole ce qu'une
   période a produit.
4. **La série temporelle** — la trajectoire complète, qui révèle les tendances.

Le même analyseur (`mesure`) a servi pour les quatre : c'est toute la force du
moteur temporel, qui découple *ce qu'on mesure* de *quand on le mesure*.